# Securing the Couchbase MCP Server with Auth0

This tutorial walks through **two OAuth flows** for the Couchbase MCP server, using Auth0 as the identity provider:

| Flow | Tested with | Style |
|------|-------------|-------|
| **Flow A — Token verify** | MCP Inspector and headless agent | Static, long-lived token; no browser interaction |
| **Flow B — Non-DCR** | MCP Inspector and VS Code| Manual client registration; browser-based login |
| **Flow C - DCR**  | VS Code and Claude Desktop | dynamic client registration with browser login, tested with VS Code|

---

## Prerequisites

- An Auth0 account (free trial at [auth0.com](https://auth0.com))
- Couchbase MCP server installed via PyPI (`uvx couchbase-mcp-server`)
- MCP Inspector (`npx @modelcontextprotocol/inspector`) or VS Code for flow 3 and 4
- A running Couchbase cluster with credentials
- `jq` installed for parsing JSON responses

---

## Part 1 — Auth0 Setup (shared by all flows)

### Step 1.1 — Note your tenant domain

After logging into the Auth0 dashboard, your tenant domain is shown at the top of the screen. It looks like:

```
your-tenant.auth0.com
```

You'll use this as `<TENANT>` throughout this tutorial.


### Step 1.2 — Create the API (represents the MCP server)

1. Go to **Applications** → **APIs** → **Create API**
2. Fill in:
   - **Name**: `Couchbase MCP Server`
   - **Identifier**: `http://127.0.0.1:8000/mcp` — this becomes the token's `aud` claim and must match `--oauth-audience` exactly. It cannot be changed after creation.
   - **Signing Algorithm**: `RS256`
3. Scroll down to **Access Policy for Applications** and set:
   - **Within user-delegated access**: `All apps allowed` — required for Flows 2 and 3 (browser login flows). Has no effect on Flow 1 since M2M uses client credentials, not user-delegated access.
   - **Within client access**: leave as `Per-app authorization` — M2M apps are explicitly authorized per-app when created in Step 2.1.
4. Click **Create**

<img src="auth0_screenshots/Auth0_1.png" width="500">

> ⚠️ The Identifier doesn't need to be a reachable URL — it's just a unique string. Using the MCP server's URL is convenient because it matches what the server advertises in its Protected Resource Metadata (needed for Flows 2 and 3).

### Step 1.3 — Add scopes

1. Open the API → **Permissions** tab
2. Add two permissions:
   - `couchbase-mcp:read` — description: "Read-only tool execution"
   - `couchbase-mcp:write` — description: "Write/mutation tool execution"

<img src="auth0_screenshots/Auth0_2.png" width="500">

### Step 1.4 — Disable RBAC token injection

1. Still in the API → **Settings** tab
2. Scroll to **RBAC Settings**
3. Leave **Enable RBAC** → **Off**
4. Leave **Add Permissions in the Access Token** → **Off**

This ensures granted scopes land in the `scope` claim (which the MCP server reads), not a separate `permissions` array.

### Step 1.5 — Collect server config values

Note these down — used in every server startup command:

| Value | Format |
|---|---|
| **Issuer** | `https://<TENANT>/` — **keep the trailing slash**, must match the token's `iss` |
| **JWKS URI** | `https://<TENANT>/.well-known/jwks.json` |
| **Token endpoint** | `https://<TENANT>/oauth/token` |
| **Audience** | `http://127.0.0.1:8000/mcp` |

---

---

## Part 3 — Flow 2: Non-DCR (Browser Login, Pre-registered Client)

This flow uses a pre-registered public client with a browser-based authorization code + PKCE flow. The user logs in interactively via Auth0's login page.

### Step 3.1 — Create a test user

1. Go to **User Management** → **Users** → **Create User**
2. Fill in email and password
3. **Connection**: `Username-Password-Authentication`
4. **Save**

<img src="auth0_screenshots/Auth0_9.png" width="500">



### Step 3.2 — Enable Resource Parameter Compatibility Profile

VS Code and MCP Inspector both send a `resource` parameter when they discover an authorization server via PRM. Auth0 must be configured to honour this parameter, otherwise it ignores it and issues a token with the wrong audience — causing `401 invalid_token` even after a successful browser login.

1. Go to **Settings** (gear icon, left nav) → **Advanced** tab
2. Find **Resource Parameter Compatibility Profile** → toggle **On** → **Save**

> 📸 *Screenshot: Resource Parameter Compatibility Profile toggle enabled*

### Step 3.3 — Create the client application

1. Go to **Applications** → **Applications** → **Create Application**
2. Name: `CB MCP Client`
3. Type: **Single Page Application**
4. Click **Create**
5. Go to **Settings** tab and add the redirect URIs for both VS Code and MCP Inspector:
   - **Allowed Callback URLs**:
     - `http://127.0.0.1:6274/oauth/callback` — MCP Inspector
     - `http://127.0.0.1:33418` — VS Code
     - `https://vscode.dev/redirect` — VS Code (web)
   - **Allowed Web Origins**:
     - `http://127.0.0.1:6274` - MCP Inspector
     - `http://127.0.0.1:33418` - VS Code

6. **Save Changes**

<img src="auth0_screenshots/Auth0_10.png" width="500">

<img src="auth0_screenshots/Auth0_11.png" width="500">


7. Copy the **Client ID** → `<SPA_CLIENT_ID>`

### Step 3.4 — Authorize the client for the API

1. Go to the application → **APIs** tab
2. Find **Couchbase MCP Server** → toggle it **On**
3. Expand it and make sure both `couchbase-mcp:read` and `couchbase-mcp:write` are visible


### Step 3.5 — Start the MCP server with PRM

```bash
uvx couchbase-mcp-server \
  --transport=http \
  --connection-string="couchbase://127.0.0.1" \
  --username="Administrator" \
  --password="<your-couchbase-password>" \
  --read-only-mode=false \
  --oauth-jwks-uri="https://<TENANT>/.well-known/jwks.json" \
  --oauth-issuer="https://<TENANT>/" \
  --oauth-audience="http://127.0.0.1:8000/mcp" \
  --oauth-mcp-base-url="http://127.0.0.1:8000"
```

The addition of `--oauth-mcp-base-url` enables the PRM endpoint, which clients use to discover Auth0 as the authorization server.

**Verify the PRM document:**

```bash
curl -s http://127.0.0.1:8000/.well-known/oauth-protected-resource/mcp | python3 -m json.tool
```

Expected response:

```json
{
  "resource": "http://127.0.0.1:8000/mcp",
  "authorization_servers": ["https://<TENANT>/"],
  "scopes_supported": ["couchbase-mcp:read", "couchbase-mcp:write"]
}
```


### Step 3.6 — Connect via MCP Inspector

```bash
npx @modelcontextprotocol/inspector
```

In the Inspector UI:

| Field | Value |
|---|---|
| **Transport Type** | Streamable HTTP |
| **URL** | `http://127.0.0.1:8000/mcp` |
| **Client ID** | `<SPA_CLIENT_ID>` |
| **Client Secret** | *(leave blank)* |
| **Redirect URL** | `http://127.0.0.1:6274/oauth/callback` |
| **Scope** | `openid couchbase-mcp:read couchbase-mcp:write` |

Click **Connect** → Auth0 login page opens → log in with the test user → accept consent → Inspector shows **"Successfully authenticated with OAuth"**.


### Step 3.7 — Connect via VS Code

In VS Code, add the MCP server to `mcp.json` with no token — VS Code will prompt for the client ID and drive the auth code + PKCE flow automatically:

```json
{
  "servers": {
    "couchbase-auth0-nondcr": {
      "type": "http",
      "url": "http://127.0.0.1:8000/mcp"
    }
  }
}
```

Click on `Start`. VS Code will prompt that `Dynamic Client Registration is not supported`, Click `Cancel`.
When VS Code prompts for a **Client ID**, enter `<SPA_CLIENT_ID>`. VS Code will prompt asking if you want to open external website, Click on `Open`. A browser window will open with the Auth0 login page → log in and authorize the app → VS Code connects.


<img src="auth0_screenshots/Auth0_12.png" width="500">

<img src="auth0_screenshots/Auth0_13.png" width="500">

<img src="auth0_screenshots/Auth0_16.png" width="500">




### Step 3.8 — Verify

**In MCP Inspector:**
1. **Tools** tab → **List Tools** — all 24 tools visible
2. Run a read tool → success
3. Run a write tool → success

<img src="auth0_screenshots/Auth0_19.png" width="500">

**In VS Code:**
1. Ask Copilot to call a Couchbase tool (e.g. "list my Couchbase buckets") → success

<img src="auth0_screenshots/Auth0_18.png" width="500">



---

## Part 4 — Flow 3: DCR (Dynamic Client Registration)

This flow requires no pre-registered client ID. The MCP host (VS Code) discovers Auth0 via the PRM document and self-registers on the fly.

### Step 4.1 — Enable Dynamic Client Registration

1. Go to **Settings** (left nav, gear icon) → **Advanced** tab
2. Find **Dynamic Client Registration** (older tenants may label it "OIDC Dynamic Application Registration")
3. Toggle it **On** → **Save**

Verify DCR is enabled:

```bash
curl -s https://<TENANT>/.well-known/openid-configuration | python3 -m json.tool | grep registration_endpoint
```

A `registration_endpoint` value (`https://<TENANT>/oidc/register`) should appear.


### Step 4.2 — Enable Resource Parameter Compatibility Profile

1. Still in **Settings** → **Advanced**
2. Find **Resource Parameter Compatibility Profile** → toggle **On** → **Save**

This makes Auth0 treat the client's `resource` parameter as the audience, issuing a standard RS256 JWT for your API. Without this, Auth0 ignores the `resource` parameter and the flow fails.


### Step 4.3 — Configure API for third-party DCR clients

DCR clients are treated as third-party apps in Auth0 and need explicit API authorization.

1. Go to **Applications** → **APIs** → **Couchbase MCP Server** → **Settings** tab
2. Scroll to **Default Permissions for Third-Party Applications**
3. Select **Authorized for User-Delegated Access**
4. Tick both `couchbase-mcp:read` and `couchbase-mcp:write`
5. **Save**


### Step 4.4 — Promote the login connection to domain-level

DCR clients can only authenticate users through domain-level connections. This requires a Management API call.

**Get the connection ID:**
1. Go to **Authentication** → **Database** → click your connection (e.g. `Username-Password-Authentication`)
2. The URL contains the connection ID (`con_...`)


**Get a Management API token:**
1. Go to **Applications** → **APIs** → **Auth0 Management API**
2. Click the **API Explorer** tab → copy the token shown there

<img src="auth0_screenshots/Auth0_20.png" width="500">

**Set the connection as domain-level:**

```bash
curl -X PATCH https://<TENANT>/api/v2/connections/<CONNECTION_ID> \
  -H "Authorization: Bearer <MGMT_TOKEN>" \
  -H "Content-Type: application/json" \
  -d '{"is_domain_connection": true}'
```

**Verify:**

```bash
curl -s https://<TENANT>/api/v2/connections/<CONNECTION_ID> \
  -H "Authorization: Bearer <MGMT_TOKEN>" | jq '.is_domain_connection'
```

Should return `true`.


### Step 4.5 — Start the MCP server

Same as Flow 2 — `--oauth-mcp-base-url` must be set:

```bash
uvx couchbase-mcp-server \
  --transport=http \
  --connection-string="couchbase://127.0.0.1" \
  --username="Administrator" \
  --password="<your-couchbase-password>" \
  --read-only-mode=false \
  --oauth-jwks-uri="https://<TENANT>/.well-known/jwks.json" \
  --oauth-issuer="https://<TENANT>/" \
  --oauth-audience="http://127.0.0.1:8000/mcp" \
  --oauth-mcp-base-url="http://127.0.0.1:8000"
```

Verify the PRM document is reachable (same curl as Step 3.4).

### Step 4.6 — Connect from VS Code

In VS Code, add the MCP server to `mcp.json` with no token and no client ID:

```json
{
  "servers": {
    "couchbase-auth0-dcr": {
      "url": "http://127.0.0.1:8000/mcp"
    }
  }
}
```

On connect, VS Code will:
1. Hit the MCP server → get a `401`
2. Read the PRM document → discover Auth0
3. `POST /oidc/register` → self-register → get a `tpc_...` client ID
4. Open a browser to Auth0's `/authorize` with `resource=http://127.0.0.1:8000/mcp`
5. You log in and consent to the scopes
6. VS Code receives the RS256 JWT → sends it to the MCP server → connected

<img src="auth0_screenshots/Auth0_21.png" width="500">

<img src="auth0_screenshots/Auth0_23.png" width="500">

<img src="auth0_screenshots/Auth0_25.png" width="500">



**Verify DCR happened:**

1. Go to **Auth0 Dashboard** → **Applications** → **Applications**
2. You should see a newly registered app with a `tpc_...` Client ID — this is proof DCR occurred

**Verify tools work:**
1. Ask Copilot Chat to call a Couchbase tool (e.g. "list my Couchbase buckets")
2. Read tool → success
3. Write tool → success

<img src="auth0_screenshots/Auth0_24.png" width="500">

### Step 4.7 - Connect from Claude Desktop

Claude Desktop only speaks stdio transport, not HTTP directly. Use `mcp-remote` as a stdio↔HTTP bridge — it handles the DCR flow automatically including opening the browser for login.

> ⚠️ No redirect URI pre-registration is needed in Auth0 for DCR — `mcp-remote` declares its own redirect URI at registration time and Auth0 accepts it dynamically.

**Make sure the MCP server is already running in the terminal** (same startup command as Step 4.5), then go to `Claude Desktop` ->  `Settings` -> `Developer` -> `Edit Config`,   add this in the `mcpServers` config in `claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "couchbase-auth0-dcr": {
      "command": "npx",
      "args": [
        "mcp-remote@latest",
        "http://127.0.0.1:8000/mcp",
        "--allow-http"
      ]
    }
  }
}
```

> ℹ️ `--allow-http` is required because the server URL is `http://` — `mcp-remote` refuses non-HTTPS endpoints without this flag.

<img src="auth0_screenshots/Auth0_30.png" width="500">

Restart Claude Desktop after saving the config. On connect, `mcp-remote` will:
1. Hit the MCP server → get a `401`
2. Read the PRM document → discover Auth0
3. `POST /oidc/register` → self-register → get a `tpc_...` client ID
4. Open a browser to Auth0's `/authorize`
5. You log in and consent to the scopes
6. Claude Desktop receives the JWT → connected

<img src="auth0_screenshots/Auth0_26.png" width="500">

<img src="auth0_screenshots/Auth0_28.png" width="500">


**Verify tools work:**
1. Ask Claude to call a Couchbase tool (e.g. "list my Couchbase buckets")
2. Read tool → success
3. Write tool → success

<img src="auth0_screenshots/Auth0_29.png" width="500">
